# 🧪 Sesión 35 · Renderizado Dinámico del DOM y Estado del Cliente

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo cambia la página sin recargarla?

---

## 🖥️ Así se verá el resultado final

```html
<ul id="lista">
  <li data-id="1">Rayuela <button>Eliminar</button></li>
  <li data-id="2">Cien años de soledad <button>Eliminar</button></li>
</ul>

<!-- al hacer clic en Eliminar del id 1 -->
<ul id="lista">
  <li data-id="2">Cien años de soledad <button>Eliminar</button></li>
</ul>

La página no se recarga: solo cambia el DOM.
```

_El estado es la fuente de verdad; el DOM es una consecuencia. Si actualizas el DOM sin actualizar el estado, la interfaz miente._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Mantener un estado del cliente.
- Renderizar una lista a partir del estado.
- Actualizar la interfaz sin recargar la página.

## ✅ Criterios de aceptación

El verificador comprobará estos criterios sobre los archivos que produzcas:

- [ ] Hay estado, render y eventos
- [ ] La página carga el renderizador

## 🧩 El reto

Implementa una lista reactiva: el estado vive en JavaScript y el DOM se vuelve a renderizar cuando cambia.

**Requisitos del contrato**

- Existe un objeto de estado con los libros.
- Existe una función render que reconstruye la lista.
- La lista se inserta con innerHTML o con nodos del DOM.
- Un botón Eliminar actualiza el estado y vuelve a renderizar.
- La página no se recarga.

> **Pista conceptual:** Si borras un elemento del DOM a mano pero no del estado, el próximo render lo revive. Cambia el estado y deja que render haga el resto.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion35.md` con la captura de la lista antes y después de eliminar.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `index.html`, `app.js`


### 📄 `index.html`

Página con el contenedor de la lista.

In [2]:
%%writefile index.html
<!DOCTYPE html>
<html lang="es">
<head><meta charset="UTF-8"><title>Biblioteca reactiva</title></head>
<body>
  <h1>Biblioteca</h1>
  <ul id="lista"></ul>
  <script src="app.js"></script>
</body>
</html>


Writing index.html


### 📄 `app.js`

Estado y renderizado.

In [3]:
%%writefile app.js
const API = "/api/libros";

// Única fuente de verdad: la interfaz se dibuja a partir de este objeto.
const estado = {
  libros: [],
  filtro: "",
  cargando: false,
  error: null,
};

const lista = document.querySelector("#lista");

// ---------- Render ----------
function render() {
  lista.replaceChildren();

  if (estado.cargando) {
    agregarItem("Cargando...");
    return;
  }
  if (estado.error) {
    agregarItem(estado.error);
    return;
  }

  const visibles = estado.libros.filter((libro) =>
    libro.titulo.toLowerCase().includes(estado.filtro.toLowerCase())
  );

  if (visibles.length === 0) {
    agregarItem("No hay libros que mostrar.");
    return;
  }

  for (const libro of visibles) {
    const item = document.createElement("li");
    // textContent evita inyectar HTML si un título trae etiquetas.
    item.textContent = libro.titulo + " (" + libro.anio + ")";
    item.dataset.id = libro.id;

    const boton = document.createElement("button");
    boton.textContent = "Eliminar";
    boton.dataset.accion = "eliminar";
    item.appendChild(boton);

    lista.appendChild(item);
  }
}

function agregarItem(texto) {
  const item = document.createElement("li");
  item.textContent = texto;
  lista.appendChild(item);
}

// ---------- Acciones que cambian el estado ----------
async function cargarLibros() {
  estado.cargando = true;
  estado.error = null;
  render();
  try {
    const respuesta = await fetch(API);
    if (!respuesta.ok) throw new Error("Error " + respuesta.status + " al cargar");
    const datos = await respuesta.json();
    estado.libros = datos.libros;
  } catch (error) {
    estado.error = error.message;
  } finally {
    estado.cargando = false;
    render();
  }
}

async function eliminarLibro(id) {
  try {
    const respuesta = await fetch(API + "/" + id, { method: "DELETE" });
    if (!respuesta.ok) throw new Error("Error " + respuesta.status + " al eliminar");
    estado.libros = estado.libros.filter((libro) => libro.id !== id);
  } catch (error) {
    estado.error = error.message;
  }
  render();
}

// ---------- Interfaz de control (filtro) ----------
function construirControles() {
  const filtro = document.createElement("input");
  filtro.placeholder = "Filtrar por título";
  filtro.addEventListener("input", (evento) => {
    estado.filtro = evento.target.value;
    render();
  });
  lista.before(filtro);
}

// ---------- Eventos ----------
// Un solo listener en la lista (delegación): sirve para los <li> que se creen después.
lista.addEventListener("click", (evento) => {
  const boton = evento.target.closest("button[data-accion='eliminar']");
  if (!boton) return;
  const id = Number(boton.closest("li").dataset.id);
  eliminarLibro(id);
});

construirControles();
cargarLibros();


Writing app.js


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [4]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.js", regex=["const\\s+estado|let\\s+estado|estado\\s*=", "function\\s+render|const\\s+render", "innerHTML|createElement|appendChild", "addEventListener", "querySelector"], descripcion="Hay estado, render y eventos"))
resultados.append(revisar_archivo("index.html", exigir=["id=\"lista\"", "app.js"], descripcion="La página carga el renderizador"))
resumen(resultados)


✅ Hay estado, render y eventos
✅ La página carga el renderizador
------------------------------------------------------------------------------
RESULTADO: 2/2 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion35.md` a `auditorias/auditoria_sesion35.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
